<a href="https://colab.research.google.com/github/Tarunk005/student-random-forest-gridsearch/blob/main/student_random_forest_gridsearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import logging
import pandas as pd
from io import StringIO
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import f1_score

In [2]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [3]:
raw_data = """study_hours,attendance_pct,assignments_done,pass
1.5,60.0,3,0
2.0,70.0,4,0
3.0,75.0,5,1
4.5,80.0,6,1
2.5,65.0,4,0
5.0,90.0,8,1
1.0,55.0,2,0
3.5,78.0,6,1
4.0,85.0,7,1
2.8,72.0,5,1
1.2,58.0,2,0
6.0,95.0,9,1
"""

In [7]:
df = pd.read_csv(StringIO(raw_data))

# Features and target
X = df.drop("pass", axis=1)
y = df["pass"]

# Step 2: Train/test split with random_state=42
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25,random_state=42,stratify=y)

# Step 3: Define parameter grid and run GridSearchCV
param_grid = {
    "n_estimators": [10, 50, 100],
    "max_depth": [3, 5, 10]
}

rf = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring="f1_weighted",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

# Step 4: Log best params and best CV F1 at INFO level
logger.info(f"Best parameters: {grid_search.best_params_}")
logger.info(f"Best CV F1 score: {grid_search.best_score_:.4f}")

# Step 5: Evaluate the best estimator on the test set exactly once
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

test_f1 = f1_score(y_test, y_pred, average="weighted")
print(f"Test Weighted F1 Score: {test_f1:.4f}")

# Step 6: Store all results in a DataFrame and display key columns
results_df = pd.DataFrame(grid_search.cv_results_)

print("\nGrid Search Results:")
print(results_df[["params", "mean_test_score", "rank_test_score"]])

print(f"\nTotal parameter combinations: {len(results_df)}")
print(f"Total fits performed: {len(results_df) * 5}")

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


Test Weighted F1 Score: 1.0000

Grid Search Results:
                                   params  mean_test_score  rank_test_score
0    {'max_depth': 3, 'n_estimators': 10}              1.0                1
1    {'max_depth': 3, 'n_estimators': 50}              1.0                1
2   {'max_depth': 3, 'n_estimators': 100}              1.0                1
3    {'max_depth': 5, 'n_estimators': 10}              1.0                1
4    {'max_depth': 5, 'n_estimators': 50}              1.0                1
5   {'max_depth': 5, 'n_estimators': 100}              1.0                1
6   {'max_depth': 10, 'n_estimators': 10}              1.0                1
7   {'max_depth': 10, 'n_estimators': 50}              1.0                1
8  {'max_depth': 10, 'n_estimators': 100}              1.0                1

Total parameter combinations: 9
Total fits performed: 45
